# Lab 8 companion notebook: the rides data in a dataframe

Run `python3 gen_data.py` in this folder first. This notebook is the same 60,000 real NYC taxi rides the harness grades, in the tool you already use. The point is the bridge: DuckDB can read a Parquet file into pandas, and it can run SQL over a pandas frame in place. You never have to choose.

Needs: `pip install duckdb pandas pyarrow matplotlib`

In [ ]:
import time
import duckdb
import pandas as pd

con = duckdb.connect()
df = con.sql("SELECT * FROM 'data/rides.parquet'").df()      # Parquet -> pandas, one line
zones = con.sql("SELECT * FROM 'data/zones.csv'").df()
df.info()

## Revenue by month

Lab question Q2, three ways. The first is pandas. The second is SQL over the *same frame*: DuckDB scans `df`'s memory directly, so `FROM df` costs no copy. The third reads the Parquet file again.

In [ ]:
by_month_pd = (df['fare'] + df['tip']).groupby(df['month']).sum().round(2)
by_month_sql = con.sql("SELECT month, round(sum(fare + tip), 2) AS revenue FROM df GROUP BY month ORDER BY month").df()
by_month_file = con.sql("SELECT month, round(sum(fare + tip), 2) AS revenue FROM 'data/rides.parquet' GROUP BY month ORDER BY month").df()
assert list(by_month_pd.values) == list(by_month_sql['revenue']) == list(by_month_file['revenue'])
by_month_sql.plot.bar(x='month', y='revenue', legend=False, title='revenue by month, 2024 sample');

## When do people tip?

Card rides only, because the meter never records cash tips (a fact about data collection, not about riders). Tip rate by pickup hour.

In [ ]:
tips = con.sql("""
    SELECT hour, round(100 * sum(tip) / sum(fare), 1) AS tip_pct, count(*) AS n
    FROM df WHERE payment = 'card' GROUP BY hour ORDER BY hour
""").df()
tips.plot(x='hour', y='tip_pct', title='tip as % of fare, card rides, by pickup hour');
tips.sort_values('tip_pct', ascending=False).head(3)

## The join (Q8) and a map you can read

`rides` is the fact table, `zones` the dimension table. One join turns zone ids into borough names.

In [ ]:
by_borough = con.sql("""
    SELECT z.Borough AS borough, count(*) AS n_rides, round(sum(r.fare + r.tip), 2) AS revenue
    FROM df r JOIN zones z ON r.pickup_zone = z.LocationID
    GROUP BY z.Borough ORDER BY revenue DESC
""").df()
by_borough

## Three GROUP BYs, timed

The same numbers as `measure_parquet.py` step 4, inside the notebook. Run the cell a few times; the first run of anything includes warm-up.

In [ ]:
def best_of(fn, n=5):
    t = float('inf')
    for _ in range(n):
        t0 = time.perf_counter(); fn(); t = min(t, time.perf_counter() - t0)
    return round(t * 1000, 2)

print('pandas groupby            ', best_of(lambda: (df['fare'] + df['tip']).groupby(df['month']).sum()), 'ms')
print('DuckDB over the frame     ', best_of(lambda: con.sql('SELECT month, sum(fare + tip) FROM df GROUP BY month').fetchall()), 'ms')
print('DuckDB over the Parquet   ', best_of(lambda: con.sql("SELECT month, sum(fare + tip) FROM 'data/rides.parquet' GROUP BY month").fetchall()), 'ms')

## The anti-pattern, in the tool where it usually happens

`slow_queries.py` pair B. Pulling everything into a frame and filtering there is how most notebooks are written. Here it is next to the filter pushed into SQL.

In [ ]:
def fetch_then_filter():
    everything = con.sql("SELECT * FROM 'data/rides.parquet'").df()     # 60,000 x 12 cross the boundary
    return round((everything.loc[everything.month == 12, 'fare'] + everything.loc[everything.month == 12, 'tip']).sum(), 2)

def filter_in_sql():
    return con.sql("SELECT round(sum(fare + tip), 2) FROM 'data/rides.parquet' WHERE month = 12").fetchone()[0]

assert fetch_then_filter() == filter_in_sql()
print('fetch everything, filter in pandas', best_of(fetch_then_filter, 3), 'ms')
print('filter in SQL                     ', best_of(filter_in_sql, 3), 'ms')

## Your turn

1. Write the tip-rate-by-hour query as a pandas expression and time both.
2. Find the ten pickup zones (not boroughs) with the highest average fare, with at least 100 rides each. Which are they, and why?
3. Take one loop from a notebook you have actually written, of the shape `for group in groups: query(group)`, and rewrite it as a single `GROUP BY` here.